# UQ-SHRED Comprehensive Experiments: Isotropic Turbulence Dataset

This notebook implements the full experimental suite:

- **E1:** Reconstruction comparison (SHRED vs UQ-SHRED)
- **E2:** Calibration evaluation
- **E3:** Uncertainty-error relationship
- **E4:** Spatial uncertainty visualization
- **E5:** Temporal forecasting with UQ
- **E6:** Ablation study (sampling size)

All metrics: Relative Error, CRPS, Coverage, Sharpness, Correlation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from sklearn.preprocessing import MinMaxScaler
import os
from datetime import datetime
import json

from processdata import load_data, TimeSeriesDataset
from models import SHRED, UQ_SHRED, UQ_Forecaster, fit, fit_uq
import uq

# Settings
device = 'cuda' if torch.cuda.is_available() else 'cpu'
dataset_name = 'ISO'
print(f'Using device: {device}')

# Results directory
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
results_dir = f'results/{dataset_name}_{timestamp}'
os.makedirs(results_dir, exist_ok=True)
print(f'Results will be saved to: {results_dir}')

# Reproducibility
np.random.seed(42)
torch.manual_seed(42)

# Log configuration
config = {
    'dataset': dataset_name,
    'device': device,
    'seed': 42,
    'timestamp': timestamp
}

## Data Preparation

In [ ]:
# Load data
num_sensors = 3
lags = 100

load_X = load_data(dataset_name)
n, m = load_X.shape
print(f'{dataset_name} data shape: {load_X.shape} (timesteps, spatial dim)')

# Random sensor locations
sensor_locations = np.random.choice(m, size=num_sensors, replace=False)
print(f'Sensor locations: {sensor_locations}')

config.update({
    'num_sensors': num_sensors,
    'lags': lags,
    'data_shape': load_X.shape,
    'sensor_locations': sensor_locations.tolist()
})

In [ ]:
# Train/valid/test split
train_indices = np.random.choice(n - lags, size=1000, replace=False)
mask = np.ones(n - lags)
mask[train_indices] = 0
valid_test_indices = np.arange(0, n - lags)[np.where(mask != 0)[0]]
valid_indices = valid_test_indices[::2]
test_indices = valid_test_indices[1::2]

print(f'Train: {len(train_indices)}, Valid: {len(valid_indices)}, Test: {len(test_indices)}')

config['split'] = {'train': len(train_indices), 'valid': len(valid_indices), 'test': len(test_indices)}

In [ ]:
# Normalize
sc = MinMaxScaler()
sc = sc.fit(load_X[train_indices])
transformed_X = sc.transform(load_X)

# Generate input sequences
all_data_in = np.zeros((n - lags, lags, num_sensors))
for i in range(len(all_data_in)):
    all_data_in[i] = transformed_X[i:i+lags, sensor_locations]

# Create tensors
train_data_in = torch.tensor(all_data_in[train_indices], dtype=torch.float32).to(device)
valid_data_in = torch.tensor(all_data_in[valid_indices], dtype=torch.float32).to(device)
test_data_in = torch.tensor(all_data_in[test_indices], dtype=torch.float32).to(device)

train_data_out = torch.tensor(transformed_X[train_indices + lags - 1], dtype=torch.float32).to(device)
valid_data_out = torch.tensor(transformed_X[valid_indices + lags - 1], dtype=torch.float32).to(device)
test_data_out = torch.tensor(transformed_X[test_indices + lags - 1], dtype=torch.float32).to(device)

train_dataset = TimeSeriesDataset(train_data_in, train_data_out)
valid_dataset = TimeSeriesDataset(valid_data_in, valid_data_out)
test_dataset = TimeSeriesDataset(test_data_in, test_data_out)

print(f'Input shape: {train_data_in.shape}')
print(f'Output shape: {train_data_out.shape}')

---
# E1: Reconstruction Comparison

In [ ]:
# Train SHRED
shred = SHRED(num_sensors, m, hidden_size=64, hidden_layers=2, l1=350, l2=400, dropout=0.01).to(device)
shred_errors = fit(shred, train_dataset, valid_dataset, batch_size=128, num_epochs=1000, lr=1e-3, verbose=True, patience=50)

In [ ]:
# SHRED test error
shred.eval()
with torch.no_grad():
    shred_recon = shred(test_dataset.X)
    shred_error = (torch.linalg.norm(shred_recon - test_dataset.Y) / torch.linalg.norm(test_dataset.Y)).item()
print(f'SHRED Test Relative Error: {shred_error:.4f}')

In [ ]:
# Train UQ-SHRED
uq_shred = UQ_SHRED(num_sensors, m, hidden_size=64, hidden_layers=2, l1=350, l2=400, dropout=0.1, noise_dim=500).to(device)
uq_errors = fit_uq(uq_shred, train_dataset, valid_dataset, batch_size=128, num_epochs=1000, lr=1e-3, verbose=True, patience=50)

In [ ]:
# UQ-SHRED test error
uq_shred.eval()
samples = uq_shred.sample(test_dataset.X, n_samples=50)
samples_np = samples.cpu().numpy()

mean_recon = samples.mean(dim=0)
median_recon_np = np.median(samples_np, axis=0)
median_recon = torch.tensor(median_recon_np, dtype=torch.float32).to(device)
std_recon = samples.std(dim=0)

uq_mean_error = (torch.linalg.norm(mean_recon - test_dataset.Y) / torch.linalg.norm(test_dataset.Y)).item()
uq_median_error = (torch.linalg.norm(median_recon - test_dataset.Y) / torch.linalg.norm(test_dataset.Y)).item()

print(f'UQ-SHRED Mean Error: {uq_mean_error:.4f}')
print(f'UQ-SHRED Median Error: {uq_median_error:.4f}')

In [ ]:
# Compute metrics
crps_score = uq.crps(samples, test_dataset.Y)
sharp = uq.sharpness(samples, conf=0.95)
cal_scores = uq.calibration_scores(samples, test_dataset.Y, levels=[0.5, 0.7, 0.9, 0.95, 0.99])

errors = torch.abs(test_dataset.Y - mean_recon).flatten().cpu().numpy()
stds = std_recon.flatten().cpu().numpy()
corr = np.corrcoef(stds, errors)[0, 1]

print('\n=== E1: Reconstruction Metrics ===')
print(f'SHRED Relative Error:       {shred_error:.4f}')
print(f'UQ-SHRED Mean Error:        {uq_mean_error:.4f}')
print(f'UQ-SHRED Median Error:      {uq_median_error:.4f}')
print(f'CRPS:                       {crps_score:.4f}')
print(f'Sharpness (95% CI):         {sharp:.4f}')
print(f'Coverage (95% CI):          {cal_scores[0.95]*100:.1f}%')
print(f'UQ-Error Correlation:       {corr:.3f}')

In [ ]:
# Inverse transform
shred_recon_np = sc.inverse_transform(shred_recon.cpu().numpy())
test_truth_np = sc.inverse_transform(test_dataset.Y.cpu().numpy())
samples_orig = np.array([sc.inverse_transform(s) for s in samples_np])
mean_orig = samples_orig.mean(axis=0)
median_orig = np.median(samples_orig, axis=0)
lower_orig = np.percentile(samples_orig, 2.5, axis=0)
upper_orig = np.percentile(samples_orig, 97.5, axis=0)

# Visualization
fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)
idx = sensor_locations[0]
t = np.arange(100)

axes[0].plot(t, test_truth_np[:100, idx], 'k-', label='Ground Truth')
axes[0].plot(t, shred_recon_np[:100, idx], 'r-', label='SHRED')
axes[0].set_ylabel('Value')
axes[0].legend()
axes[0].set_title('SHRED (Deterministic)')

axes[1].fill_between(t, lower_orig[:100, idx], upper_orig[:100, idx], alpha=0.3, color='blue', label='95% CI')
axes[1].plot(t, test_truth_np[:100, idx], 'k-', label='Ground Truth')
axes[1].plot(t, mean_orig[:100, idx], 'b-', label='UQ Mean')
axes[1].plot(t, median_orig[:100, idx], 'g--', label='UQ Median')
axes[1].set_ylabel('Value')
axes[1].legend()
axes[1].set_title('UQ-SHRED (With Uncertainty)')

axes[2].plot(t, test_truth_np[:100, idx], 'k-', linewidth=1.5, label='Ground Truth')
axes[2].plot(t, shred_recon_np[:100, idx], 'r-', linewidth=1.2, label='SHRED')
axes[2].plot(t, mean_orig[:100, idx], 'b-', linewidth=1.2, label='UQ Mean')
axes[2].plot(t, median_orig[:100, idx], 'g--', linewidth=1.2, label='UQ Median')
axes[2].fill_between(t, lower_orig[:100, idx], upper_orig[:100, idx], alpha=0.15, color='blue')
axes[2].set_xlabel('Time')
axes[2].set_ylabel('Value')
axes[2].legend()
axes[2].set_title('Overlay Comparison')

plt.tight_layout()
plt.savefig(f'{results_dir}/E1_reconstruction_comparison.png', dpi=150)
plt.show()

---
# E2: Calibration

In [ ]:
print('\n=== E2: Calibration Scores ===')
print('Coverage (expected → observed):')
for level, obs in cal_scores.items():
    print(f'  {level*100:.0f}% CI → {obs*100:.1f}% coverage')

fig, ax = plt.subplots(figsize=(6, 6))
uq.plot_calibration(samples, test_dataset.Y, ax=ax)
ax.set_title('E2: UQ-SHRED Calibration')
plt.tight_layout()
plt.savefig(f'{results_dir}/E2_calibration_diagram.png', dpi=150)
plt.show()

---
# E3: Uncertainty–Error Relationship

In [ ]:
idx_sample = np.random.choice(len(errors), min(5000, len(errors)), replace=False)

fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(stds[idx_sample], errors[idx_sample], alpha=0.1, s=1)

z = np.polyfit(stds[idx_sample], errors[idx_sample], 1)
p = np.poly1d(z)
x_line = np.linspace(stds.min(), stds.max(), 100)
ax.plot(x_line, p(x_line), 'r-', linewidth=2, label='Linear fit')

ax.set_xlabel('Predicted Uncertainty (σ)', fontsize=12)
ax.set_ylabel('Actual Error |y - ŷ|', fontsize=12)
ax.set_title(f'E3: Uncertainty vs Error (ρ={corr:.3f})', fontsize=14)
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{results_dir}/E3_uncertainty_vs_error.png', dpi=150)
plt.show()

print(f'\n=== E3: Correlation ρ = {corr:.4f} ===')

---
# E4: Spatial Uncertainty

In [ ]:
# Compute spatial average uncertainty
avg_std = std_recon.cpu().numpy().mean(axis=0)  # Average over test set
spatial_dim = int(np.sqrt(m))  # Assume square grid (350x350)

# Reshape for visualization
try:
    spatial_std_2d = avg_std.reshape(spatial_dim, spatial_dim)
    mean_snapshot_2d = mean_orig[0].reshape(spatial_dim, spatial_dim)
    truth_snapshot_2d = test_truth_np[0].reshape(spatial_dim, spatial_dim)
    
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    
    im1 = axes[0].imshow(truth_snapshot_2d, aspect='auto', cmap='coolwarm')
    axes[0].set_title('Ground Truth (t=0)')
    axes[0].axis('off')
    plt.colorbar(im1, ax=axes[0], fraction=0.046)
    
    im2 = axes[1].imshow(mean_snapshot_2d, aspect='auto', cmap='coolwarm')
    axes[1].set_title('UQ-SHRED Mean (t=0)')
    axes[1].axis('off')
    plt.colorbar(im2, ax=axes[1], fraction=0.046)
    
    im3 = axes[2].imshow(spatial_std_2d, aspect='auto', cmap='viridis')
    axes[2].set_title('E4: Average Predictive Uncertainty (σ)')
    axes[2].axis('off')
    plt.colorbar(im3, ax=axes[2], fraction=0.046)
    
    plt.tight_layout()
    plt.savefig(f'{results_dir}/E4_spatial_uncertainty.png', dpi=150)
    plt.show()
except:
    # Fallback: plot 1D
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    axes[0].plot(mean_orig[0], alpha=0.7)
    axes[0].set_title('Mean Reconstruction (t=0)')
    axes[0].set_xlabel('Spatial Index')
    
    axes[1].plot(avg_std, alpha=0.7, color='purple')
    axes[1].set_title('E4: Average Predictive Uncertainty')
    axes[1].set_xlabel('Spatial Index')
    axes[1].set_ylabel('σ')
    
    plt.tight_layout()
    plt.savefig(f'{results_dir}/E4_spatial_uncertainty.png', dpi=150)
    plt.show()

print('\n=== E4: Spatial Uncertainty Statistics ===')
print(f'Mean uncertainty: {avg_std.mean():.4f}')
print(f'Max uncertainty:  {avg_std.max():.4f}')
print(f'Min uncertainty:  {avg_std.min():.4f}')

---
# E5: Temporal Forecasting

In [ ]:
# Prepare forecaster data
sensor_data = transformed_X[:, sensor_locations]
forecast_in = np.zeros((n - lags, lags, num_sensors))
forecast_out = np.zeros((n - lags, num_sensors))
for i in range(n - lags):
    forecast_in[i] = sensor_data[i:i+lags]
    forecast_out[i] = sensor_data[i+lags]

train_fc_in = torch.tensor(forecast_in[train_indices], dtype=torch.float32).to(device)
train_fc_out = torch.tensor(forecast_out[train_indices], dtype=torch.float32).to(device)
valid_fc_in = torch.tensor(forecast_in[valid_indices], dtype=torch.float32).to(device)
valid_fc_out = torch.tensor(forecast_out[valid_indices], dtype=torch.float32).to(device)

train_fc_dataset = TimeSeriesDataset(train_fc_in, train_fc_out)
valid_fc_dataset = TimeSeriesDataset(valid_fc_in, valid_fc_out)

# Train deterministic forecaster
print('Training SHRED forecaster...')
shred_forecaster = SHRED(num_sensors, num_sensors, hidden_size=64, hidden_layers=2, l1=100, l2=150, dropout=0.1).to(device)
fc_shred_errors = fit(shred_forecaster, train_fc_dataset, valid_fc_dataset, batch_size=64, num_epochs=200, lr=1e-3, verbose=True, patience=5)

In [ ]:
# Train UQ-Forecaster
print('Training UQ-Forecaster...')
forecaster = UQ_Forecaster(input_size=num_sensors, hidden_size=64, hidden_layers=2, noise_dim=50).to(device)
fc_errors = fit_uq(forecaster, train_fc_dataset, valid_fc_dataset, batch_size=64, num_epochs=200, lr=1e-3, verbose=True, patience=5)

In [ ]:
# Generate forecasts
horizon = 50
initial = test_data_in[0:1]

shred_forecaster.eval()
with torch.no_grad():
    history = initial.clone()
    shred_traj = []
    for t in range(horizon):
        next_val = shred_forecaster(history)
        shred_traj.append(next_val.cpu().numpy().squeeze())
        history = torch.cat([history[:, 1:, :], next_val.unsqueeze(1)], dim=1)
shred_traj = np.array(shred_traj)

uq_traj_samples = forecaster.sample_trajectory(initial, horizon=horizon, n_samples=50)
uq_traj_samples = uq_traj_samples.squeeze(1).cpu().numpy()

mean_traj = uq_traj_samples.mean(axis=0)
median_traj = np.median(uq_traj_samples, axis=0)
std_traj = uq_traj_samples.std(axis=0)
lower_traj = np.percentile(uq_traj_samples, 2.5, axis=0)
upper_traj = np.percentile(uq_traj_samples, 97.5, axis=0)

start_idx = test_indices[0] + lags
gt_future = sensor_data[start_idx:start_idx+horizon]

shred_fc_error = np.linalg.norm(shred_traj - gt_future) / np.linalg.norm(gt_future)
uq_mean_fc_error = np.linalg.norm(mean_traj - gt_future) / np.linalg.norm(gt_future)
uq_median_fc_error = np.linalg.norm(median_traj - gt_future) / np.linalg.norm(gt_future)

print('\n=== E5: Forecast Errors ===')
print(f'SHRED:     {shred_fc_error:.4f}')
print(f'UQ Mean:   {uq_mean_fc_error:.4f}')
print(f'UQ Median: {uq_median_fc_error:.4f}')

In [ ]:
# Forecast visualization
fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)
t = np.arange(horizon)
sensor_idx = 0

axes[0].plot(t, gt_future[:, sensor_idx], 'k--', linewidth=1.5, label='Ground Truth')
axes[0].plot(t, shred_traj[:, sensor_idx], 'r-', linewidth=1.2, label='SHRED')
axes[0].set_ylabel('Value')
axes[0].legend()
axes[0].set_title('SHRED Forecast')

axes[1].fill_between(t, lower_traj[:, sensor_idx], upper_traj[:, sensor_idx], alpha=0.3, color='blue', label='95% CI')
axes[1].plot(t, gt_future[:, sensor_idx], 'k--', linewidth=1.5, label='Ground Truth')
axes[1].plot(t, mean_traj[:, sensor_idx], 'b-', linewidth=1.2, label='UQ Mean')
axes[1].plot(t, median_traj[:, sensor_idx], 'g--', linewidth=1.2, label='UQ Median')
axes[1].set_ylabel('Value')
axes[1].legend()
axes[1].set_title('UQ-Forecaster')

axes[2].plot(t, gt_future[:, sensor_idx], 'k-', linewidth=1.5, label='Ground Truth')
axes[2].plot(t, shred_traj[:, sensor_idx], 'r-', linewidth=1.2, label='SHRED')
axes[2].plot(t, mean_traj[:, sensor_idx], 'b-', linewidth=1.2, label='UQ Mean')
axes[2].fill_between(t, lower_traj[:, sensor_idx], upper_traj[:, sensor_idx], alpha=0.15, color='blue')
axes[2].set_xlabel('Forecast Horizon')
axes[2].set_ylabel('Value')
axes[2].legend()
axes[2].set_title('E5: Overlay')

plt.tight_layout()
plt.savefig(f'{results_dir}/E5_forecast_comparison.png', dpi=150)
plt.show()

In [ ]:
# E5: Uncertainty growth over horizon
avg_std_horizon = std_traj.mean(axis=1)
shred_error_horizon = np.abs(shred_traj - gt_future).mean(axis=1)
uq_error_horizon = np.abs(mean_traj - gt_future).mean(axis=1)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(np.arange(horizon), avg_std_horizon, 'b--', marker='o', markersize=3, label='UQ Uncertainty (σ)')
ax.plot(np.arange(horizon), shred_error_horizon, 'r-', marker='s', markersize=3, alpha=0.7, label='SHRED Error')
ax.plot(np.arange(horizon), uq_error_horizon, 'b-', marker='s', markersize=3, alpha=0.7, label='UQ Mean Error')
ax.set_xlabel('Forecast Horizon')
ax.set_ylabel('Value')
ax.set_title('E5: Uncertainty and Error Growth Over Horizon')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{results_dir}/E5_uncertainty_growth.png', dpi=150)
plt.show()

---
# E6: Ablation Study

In [ ]:
n_samples_list = [10, 25, 50]
ablation_results = []

print('\n=== E6: Ablation Study ===')
for n_samp in n_samples_list:
    print(f'\nn_samples={n_samp}...')
    samp = uq_shred.sample(test_dataset.X, n_samples=n_samp)
    
    mean_pred = samp.mean(dim=0)
    rel_error = (torch.linalg.norm(mean_pred - test_dataset.Y) / torch.linalg.norm(test_dataset.Y)).item()
    crps = uq.crps(samp, test_dataset.Y)
    sharp_val = uq.sharpness(samp, conf=0.95)
    cal = uq.calibration_scores(samp, test_dataset.Y, levels=[0.95])[0.95]
    
    ablation_results.append({
        'n_samples': n_samp,
        'rel_error': rel_error,
        'crps': crps,
        'sharpness': sharp_val,
        'coverage_95': cal
    })
    print(f'  Error: {rel_error:.4f}, CRPS: {crps:.4f}, Sharp: {sharp_val:.4f}, Cov: {cal*100:.1f}%')

n_samp_arr = np.array([r['n_samples'] for r in ablation_results])
rel_err_arr = np.array([r['rel_error'] for r in ablation_results])
crps_arr = np.array([r['crps'] for r in ablation_results])
sharp_arr = np.array([r['sharpness'] for r in ablation_results])
cov_arr = np.array([r['coverage_95'] for r in ablation_results]) * 100

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

axes[0, 0].plot(n_samp_arr, rel_err_arr, 'o-', linewidth=2, markersize=8)
axes[0, 0].axhline(shred_error, color='r', linestyle='--', label='SHRED')
axes[0, 0].set_xlabel('Number of Samples')
axes[0, 0].set_ylabel('Relative Error')
axes[0, 0].set_title('Accuracy')
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3)

axes[0, 1].plot(n_samp_arr, crps_arr, 'o-', linewidth=2, markersize=8, color='green')
axes[0, 1].set_xlabel('Number of Samples')
axes[0, 1].set_ylabel('CRPS')
axes[0, 1].set_title('CRPS')
axes[0, 1].grid(alpha=0.3)

axes[1, 0].plot(n_samp_arr, sharp_arr, 'o-', linewidth=2, markersize=8, color='purple')
axes[1, 0].set_xlabel('Number of Samples')
axes[1, 0].set_ylabel('Sharpness')
axes[1, 0].set_title('Sharpness')
axes[1, 0].grid(alpha=0.3)

axes[1, 1].plot(n_samp_arr, cov_arr, 'o-', linewidth=2, markersize=8, color='orange')
axes[1, 1].axhline(95, color='k', linestyle='--', label='Nominal')
axes[1, 1].set_xlabel('Number of Samples')
axes[1, 1].set_ylabel('Coverage (%)')
axes[1, 1].set_title('Calibration')
axes[1, 1].legend()
axes[1, 1].grid(alpha=0.3)

plt.suptitle('E6: Ablation Study', fontsize=14, y=1.00)
plt.tight_layout()
plt.savefig(f'{results_dir}/E6_ablation.png', dpi=150)
plt.show()

---
# Summary

In [ ]:
summary_lines = []
summary_lines.append('=' * 90)
summary_lines.append('UQ-SHRED COMPREHENSIVE RESULTS')
summary_lines.append('=' * 90)
summary_lines.append(f'Dataset: {dataset_name}')
summary_lines.append(f'Timestamp: {timestamp}')
summary_lines.append('')
summary_lines.append('E1: RECONSTRUCTION')
summary_lines.append(f'{"Metric":<30} {"SHRED":<15} {"UQ Mean":<15} {"UQ Median":<15}')
summary_lines.append('-' * 90)
summary_lines.append(f'{"Relative Error":<30} {shred_error:<15.4f} {uq_mean_error:<15.4f} {uq_median_error:<15.4f}')
summary_lines.append(f'{"CRPS":<30} {"---":<15} {crps_score:<15.4f} {"---":<15}')
summary_lines.append(f'{"Sharpness":<30} {"---":<15} {sharp:<15.4f} {"---":<15}')
summary_lines.append(f'{"Coverage (95%)":<30} {"---":<15} {cal_scores[0.95]*100:<14.1f}% {"---":<15}')
summary_lines.append(f'{"Correlation":<30} {"---":<15} {corr:<15.3f} {"---":<15}')
summary_lines.append('')
summary_lines.append('E5: FORECASTING')
summary_lines.append(f'{"Forecast Error":<30} {shred_fc_error:<15.4f} {uq_mean_fc_error:<15.4f} {uq_median_fc_error:<15.4f}')
summary_lines.append('')
summary_lines.append('E6: ABLATION')
summary_lines.append(f'{"n_samples":<12} {"Error":<12} {"CRPS":<12} {"Sharp":<12} {"Coverage":<12}')
for r in ablation_results:
    summary_lines.append(f'{r["n_samples"]:<12} {r["rel_error"]:<12.4f} {r["crps"]:<12.4f} {r["sharpness"]:<12.4f} {r["coverage_95"]*100:<11.1f}%')
summary_lines.append('=' * 90)

summary_text = '\n'.join(summary_lines)
print(summary_text)

with open(f'{results_dir}/metrics.txt', 'w') as f:
    f.write(summary_text)
with open(f'{results_dir}/config.json', 'w') as f:
    json.dump(config, f, indent=2)
with open(f'{results_dir}/ablation_results.json', 'w') as f:
    json.dump(ablation_results, f, indent=2)

print(f'\n✓ Results saved to: {results_dir}/')